In [1]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install","delta-spark==3.1.0",
                "python-dotenv", "psycopg2-binary", 
                "pymongo", 
               ], check=True)

print("Installation done 👍")


Installation done 👍


In [2]:
# ─────────────────────────────────────────────────────────────
# Cell 2 — Configuration
#
# All secrets loaded from .env file
# No credentials visible in notebook code
# Validation check at the end — fails immediately
# if any required secret is missing
#
# In Azure this pattern becomes:
#   dbutils.secrets.get(scope="atlas", key="postgres-password")
#   All secrets stored in Azure Key Vault
#   No credentials anywhere in notebook code
# ─────────────────────────────────────────────────────────────

import os
import uuid
import logging
from datetime import datetime, date
from dotenv import load_dotenv

# Load all secrets from .env file
# Explicit path — works regardless of working directory
load_dotenv("/home/jovyan/work/.env", override=True)

# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# All values loaded from environment variables
# os.getenv(key, default) — default only used if key missing
# For secrets the default is empty string ""
# Validation below catches empty secrets before they cause
# confusing errors deep inside Spark or JDBC
# ─────────────────────────────────────────────────────────────
CONFIG = {
    # ── MinIO — local Azure Data Lake Storage ─────────────────
    # S3-compatible object storage
    # Stores Silver Delta Lake files that Gold reads
    "minio_endpoint":    os.getenv("MINIO_ENDPOINT",  "http://minio:9000"),
    "minio_access_key":  os.getenv("MINIO_ACCESS_KEY", ""),
    "minio_secret_key":  os.getenv("MINIO_SECRET_KEY", ""),

    # ── Bucket names ──────────────────────────────────────────
    "silver_bucket":     os.getenv("SILVER_BUCKET",   "silver"),
    "gold_bucket":       os.getenv("GOLD_BUCKET",     "gold"),

    # ── PostgreSQL — local Synapse Analytics ──────────────────
    # Gold writes pre-aggregated metrics here
    # Power BI connects to PostgreSQL for dashboards
    # In Azure replace with Synapse connection details
    # All values from .env — no credentials in code
    "postgres_host":     os.getenv("POSTGRES_HOST"),
    "postgres_port":     os.getenv("POSTGRES_PORT"),
    "postgres_db":       os.getenv("POSTGRES_DB"),
    "postgres_user":     os.getenv("POSTGRES_USER"),
    "postgres_password": os.getenv("POSTGRES_PASSWORD"),

    # ── MongoDB — local Cosmos DB ─────────────────────────────
    # Audit log — records every Gold job run
    "mongodb_uri": os.getenv("MONGODB_URI"),
    "mongodb_db":  os.getenv("MONGODB_DB"),

    # ── Pipeline identity ─────────────────────────────────────
    "pipeline_name":     os.getenv("PIPELINE_NAME", "atlas-gold-aggregation"),
    "processing_date":   str(date.today()),

    # ── Gold settings ─────────────────────────────────────────
    # Only transactions with fraud_score above this
    # threshold appear in the Gold fraud_indicators table
    # 0 means all transactions with any fraud signal
    "fraud_score_threshold": 0,
}

# ─────────────────────────────────────────────────────────────
# UNIQUE RUN ID
# Stamped on every Gold record and audit log entry
# Links Gold records back to the job that produced them
# Enables debugging — find all records from one run
# ─────────────────────────────────────────────────────────────
RUN_ID = str(uuid.uuid4())

# ─────────────────────────────────────────────────────────────
# POSTGRESQL JDBC CONNECTION
# Spark uses JDBC to write DataFrames to PostgreSQL
# JDBC URL format: jdbc:postgresql://host:port/database
# POSTGRES_PROPS passed to spark.write.jdbc()
#
# In Azure replace with Synapse JDBC URL:
# jdbc:sqlserver://atlas.sql.azuresynapse.net:1433;
# database=atlas_gold;encrypt=true;
# ─────────────────────────────────────────────────────────────
POSTGRES_JDBC_URL = (
    f"jdbc:postgresql://"
    f"{CONFIG['postgres_host']}:"
    f"{CONFIG['postgres_port']}/"
    f"{CONFIG['postgres_db']}"
)

# Properties passed to every JDBC write operation
# Password comes from CONFIG which came from .env
# Never hardcoded here
POSTGRES_PROPS = {
    "user":     CONFIG["postgres_user"],
    "password": CONFIG["postgres_password"],
    "driver":   "org.postgresql.Driver",
}

# ─────────────────────────────────────────────────────────────
# VALIDATE ALL REQUIRED SECRETS ARE LOADED
# Fail immediately with a clear message if any are missing
# Better to crash here than get a confusing authentication
# error deep inside a Spark job 10 minutes later
# ─────────────────────────────────────────────────────────────
missing = [
    key for key, value in {
        "MINIO_ACCESS_KEY":  CONFIG["minio_access_key"],
        "MINIO_SECRET_KEY":  CONFIG["minio_secret_key"],
        "POSTGRES_HOST":     CONFIG["postgres_host"],
        "POSTGRES_PORT":     CONFIG["postgres_port"],
        "POSTGRES_DB":       CONFIG["postgres_db"],
        "POSTGRES_USER":     CONFIG["postgres_user"],
        "POSTGRES_PASSWORD": CONFIG["postgres_password"],
    }.items()
    if not value
]

if missing:
    raise ValueError(
        f"Missing required environment variables: {missing}\n"
        f"Check /home/jovyan/work/.env exists and contains these values\n"
        f"Run: docker cp .env atlas-jupyter:/home/jovyan/work/.env"
    )

# ─────────────────────────────────────────────────────────────
# PRINT CONFIG — mask all secrets
# Show enough to confirm correct values loaded
# Never print full secret — first 3 chars only
# ─────────────────────────────────────────────────────────────
print(f"Pipeline:         {CONFIG['pipeline_name']}")
print(f"Run ID:           {RUN_ID}")
print(f"Processing date:  {CONFIG['processing_date']}")
print()
print(f"MinIO endpoint:   {CONFIG['minio_endpoint']}")
print(f"MinIO key:        {CONFIG['minio_access_key'][:3]}***")
print(f"Silver bucket:    {CONFIG['silver_bucket']}")
print(f"Gold bucket:      {CONFIG['gold_bucket']}")
print()
print(f"PostgreSQL host:  {CONFIG['postgres_host']}")
print(f"PostgreSQL db:    {CONFIG['postgres_db']}")
print(f"PostgreSQL user:  {CONFIG['postgres_user']}")
print(f"PostgreSQL pass:  {CONFIG['postgres_password'][:3]}***")
print(f"JDBC URL:         {POSTGRES_JDBC_URL}")
print()
print(f"MongoDB URI:      {CONFIG['mongodb_uri'][:35]}...")
print()
print("All secrets loaded from .env — none visible in code")
print("Configuration ready")

Pipeline:         atlas-gold-aggregation
Run ID:           9a1110bb-38e6-4cd8-8014-9b2189af56fb
Processing date:  2026-04-13

MinIO endpoint:   http://minio:9000
MinIO key:        atl***
Silver bucket:    silver
Gold bucket:      gold

PostgreSQL host:  postgres
PostgreSQL db:    atlas_gold
PostgreSQL user:  atlasadmin
PostgreSQL pass:  atl***
JDBC URL:         jdbc:postgresql://postgres:5432/atlas_gold

MongoDB URI:      mongodb://atlasadmin:atlaspassword1...

All secrets loaded from .env — none visible in code
Configuration ready


In [3]:
import os
import time
from pyspark.sql import SparkSession

# Download PostgreSQL JDBC driver so Spark can write to PostgreSQL
# Also includes Delta Lake and Hadoop AWS for MinIO connection
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "io.delta:delta-spark_2.12:3.1.0,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.367,"
    "org.postgresql:postgresql:42.6.0 "
    "pyspark-shell"
)

# Stop any existing session — only one allowed at a time
try:
    SparkSession.getActiveSession().stop()
    time.sleep(2)
    print("Stopped existing session")
except:
    pass

spark = (
    SparkSession.builder
    .appName("Atlas — Gold Aggregation")
    .master("local[2]")

    # Delta Lake support
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")

    # Gold does heavy GROUP BY — needs more memory than Bronze/Silver
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.executor.memory",        "2g")
    .config("spark.driver.memory",          "2g")

    # Keep session alive during long aggregations
    .config("spark.network.timeout",            "3600s")
    .config("spark.executor.heartbeatInterval",  "60s")

    # MinIO connection — all credentials from CONFIG not hardcoded
    .config("spark.hadoop.fs.s3a.endpoint",
            CONFIG["minio_endpoint"])
    .config("spark.hadoop.fs.s3a.access.key",
            CONFIG["minio_access_key"])
    .config("spark.hadoop.fs.s3a.secret.key",
            CONFIG["minio_secret_key"])
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")

    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version:   {spark.version}")
print(f"App name:        {spark.sparkContext.appName}")
print(f"Spark master:    {spark.sparkContext.master}")
print(f"Executor memory: {spark.conf.get('spark.executor.memory')}")
print(f"Spark UI:        http://localhost:4040")
print("Spark session ready")

Spark version:   3.5.0
App name:        Atlas — Gold Aggregation
Spark master:    local[2]
Executor memory: 2g
Spark UI:        http://localhost:4040
Spark session ready


In [4]:
# ─────────────────────────────────────────────────────────────
# Cell 4 — Read Silver and compute Gold aggregations
#
# Gold reads Silver as a batch — not a stream
# Runs every 5 minutes and processes all Silver data
# Produces three aggregation tables written to PostgreSQL
# ─────────────────────────────────────────────────────────────

from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# Silver Delta Lake path — where we read from
SILVER_DELTA_PATH = f"s3a://{CONFIG['silver_bucket']}/delta/transactions/"

print("Reading Silver Delta Lake...")

try:
    silver_df = spark.read.format("delta").load(SILVER_DELTA_PATH)
    total     = silver_df.count()
    print(f"Silver records available: {total}")
    print()
except Exception as e:
    raise RuntimeError(
        f"Cannot read Silver Delta Lake: {e}\n"
        f"Make sure Silver stream is running in notebook 03"
    )

# ─────────────────────────────────────────────────────────────
# AGGREGATION 1 — Daily customer spending
#
# Answers: how much did each customer spend today?
# Which category do they spend most in?
# Used by: fraud detection, customer analytics, Power BI
# ─────────────────────────────────────────────────────────────
print("Computing daily customer spending...")

daily_spending_df = (
    silver_df
    .groupBy("processing_date", "customer_id", "bank_code")
    .agg(
        # Total spend in EUR — normalised across all currencies
        F.round(F.sum("amount_eur"),   2).alias("total_amount_eur"),

        # Number of transactions this customer made today
        F.count("*").alias("transaction_count"),

        # Average transaction value — useful for anomaly detection
        F.round(F.avg("amount_eur"),   2).alias("avg_transaction_eur"),

        # Largest single transaction today
        F.round(F.max("amount_eur"),   2).alias("max_transaction_eur"),

        # Spending by category — useful for customer profiling
        F.round(F.sum(
            F.when(F.col("merchant_category") == "grocery",
                   F.col("amount_eur")).otherwise(0)
        ), 2).alias("grocery_spend_eur"),

        F.round(F.sum(
            F.when(F.col("merchant_category") == "ecommerce",
                   F.col("amount_eur")).otherwise(0)
        ), 2).alias("ecommerce_spend_eur"),

        F.round(F.sum(
            F.when(F.col("merchant_category") == "fuel",
                   F.col("amount_eur")).otherwise(0)
        ), 2).alias("fuel_spend_eur"),

        F.round(F.sum(
            F.when(F.col("merchant_category") == "entertainment",
                   F.col("amount_eur")).otherwise(0)
        ), 2).alias("entertainment_spend_eur"),

        F.round(F.sum(
            F.when(F.col("merchant_category") == "transport",
                   F.col("amount_eur")).otherwise(0)
        ), 2).alias("transport_spend_eur"),

        # Top spending category for this customer today
        # Uses the category with the highest total spend
        F.first("merchant_category").alias("top_category"),

        # Pipeline metadata — links to audit log
        F.lit(RUN_ID).alias("pipeline_run_id"),
        F.current_timestamp().alias("computed_at")
    )
    .orderBy("total_amount_eur", ascending=False)
)

spending_count = daily_spending_df.count()
print(f"Daily spending aggregations: {spending_count} customer-day records")
print()
print("Top 5 spenders today:")
daily_spending_df.select(
    "customer_id",
    "bank_code",
    "total_amount_eur",
    "transaction_count",
    "top_category"
).show(5, truncate=False)

# ─────────────────────────────────────────────────────────────
# AGGREGATION 2 — Fraud indicators
#
# Answers: which transactions look suspicious today?
# What fraud signals were triggered?
# Used by: fraud team alerts, compliance reporting
# ─────────────────────────────────────────────────────────────
print("Computing fraud indicators...")

fraud_df = (
    silver_df
    # Only include transactions with at least one fraud signal
    .filter(F.col("fraud_score") > CONFIG["fraud_score_threshold"])
    .select(
        "transaction_id",
        "customer_id",
        "bank_code",
        "processing_date",
        "amount",
        "amount_eur",
        "currency",
        "merchant_name",
        "merchant_category",
        "is_large_amount",
        "is_suspicious_ip",
        "is_unknown_merchant",
        "is_pending_status",
        "is_unusual_currency",
        "fraud_score",
        "risk_level",
        "ip_address",
        "event_timestamp",
        F.lit(RUN_ID).alias("pipeline_run_id"),
        F.current_timestamp().alias("computed_at")
    )
    .orderBy("fraud_score", ascending=False)
)

fraud_count = fraud_df.count()
print(f"Flagged transactions: {fraud_count}")
print()

if fraud_count > 0:
    print("Fraud breakdown by risk level:")
    fraud_df.groupBy("risk_level") \
        .agg(F.count("*").alias("count")) \
        .orderBy("risk_level") \
        .show()

    print("Sample flagged transactions:")
    fraud_df.select(
        "transaction_id",
        "bank_code",
        "amount_eur",
        "fraud_score",
        "risk_level",
        "is_large_amount",
        "is_suspicious_ip",
        "is_unknown_merchant"
    ).show(5, truncate=False)

# ─────────────────────────────────────────────────────────────
# AGGREGATION 3 — Daily merchant summary
#
# Answers: which merchants processed the most today?
# What is the average transaction per merchant?
# Used by: merchant analytics, business development
# ─────────────────────────────────────────────────────────────
print("Computing daily merchant summary...")

merchant_df = (
    silver_df
    # Exclude unknown merchants from summary
    .filter(F.col("merchant_category") != "unknown")
    .groupBy(
        "processing_date",
        "merchant_id",
        "merchant_name",
        "merchant_category"
    )
    .agg(
        # Total revenue this merchant processed today
        F.round(F.sum("amount_eur"),   2).alias("total_revenue_eur"),

        # Number of transactions
        F.count("*").alias("transaction_count"),

        # Unique customers who visited this merchant today
        F.countDistinct("customer_id").alias("unique_customers"),

        # Average transaction value
        F.round(F.avg("amount_eur"),   2).alias("avg_transaction_eur"),

        F.lit(RUN_ID).alias("pipeline_run_id"),
        F.current_timestamp().alias("computed_at")
    )
    .orderBy("total_revenue_eur", ascending=False)
)

merchant_count = merchant_df.count()
print(f"Merchant aggregations: {merchant_count} merchant-day records")
print()
print("Top 5 merchants by revenue today:")
merchant_df.select(
    "merchant_name",
    "merchant_category",
    "total_revenue_eur",
    "transaction_count",
    "unique_customers"
).show(5, truncate=False)

print("=" * 55)
print("All three Gold aggregations computed")
print(f"  Daily spending:   {spending_count} records")
print(f"  Fraud indicators: {fraud_count} records")
print(f"  Merchant summary: {merchant_count} records")
print("Ready for Cell 5 — write to PostgreSQL")

Reading Silver Delta Lake...
Silver records available: 964

Computing daily customer spending...
Daily spending aggregations: 148 customer-day records

Top 5 spenders today:
+--------------+---------+----------------+-----------------+-------------+
|customer_id   |bank_code|total_amount_eur|transaction_count|top_category |
+--------------+---------+----------------+-----------------+-------------+
|CUST-MBANK-043|MBANK    |75698.74        |4                |entertainment|
|CUST-MBANK-003|MBANK    |73793.03        |6                |unknown      |
|CUST-PKO-033  |PKO      |68425.40        |3                |unknown      |
|CUST-ING-024  |ING      |65449.71        |13               |entertainment|
|CUST-PKO-030  |PKO      |56737.64        |7                |unknown      |
+--------------+---------+----------------+-----------------+-------------+
only showing top 5 rows

Computing fraud indicators...
Flagged transactions: 57

Fraud breakdown by risk level:
+----------+-----+
|risk_level

In [7]:
# ─────────────────────────────────────────────────────────────
# Cell 5 — Write Gold aggregations to PostgreSQL
#
# Three tables get written:
#   gold.daily_customer_spending
#   gold.fraud_indicators
#   gold.daily_merchant_summary
#
# We use mode("append") with a delete-first pattern
# This is the UPSERT equivalent for Spark JDBC
#
# Why not mode("overwrite")?
#   Overwrite drops and recreates the entire table
#   Loses all historical data from previous runs
#
# Why not mode("append") directly?
#   Appending without deleting first creates duplicates
#   Running Gold twice would double every record
#
# The pattern:
#   1. Delete today's records from PostgreSQL
#   2. Append today's fresh aggregations
#   This gives us idempotent writes — safe to rerun
# ─────────────────────────────────────────────────────────────

import psycopg2
from pymongo import MongoClient
from datetime import datetime

# ─────────────────────────────────────────────────────────────
# HELPER — delete today's records before writing
# Prevents duplicates if Gold job runs more than once today
# ─────────────────────────────────────────────────────────────
def delete_todays_records(table: str):
    """
    Delete today's records from a Gold PostgreSQL table
    before writing fresh aggregations.
    Safe to call even if no records exist for today.
    """
    conn = psycopg2.connect(
        host=CONFIG["postgres_host"],
        port=CONFIG["postgres_port"],
        dbname=CONFIG["postgres_db"],
        user=CONFIG["postgres_user"],
        password=CONFIG["postgres_password"]
    )
    try:
        cur = conn.cursor()
        cur.execute(
            f"DELETE FROM {table} WHERE transaction_date = %s",
            (CONFIG["processing_date"],)
        )
        deleted = cur.rowcount
        conn.commit()
        print(f"  Deleted {deleted} existing records from {table}")
    finally:
        conn.close()


# ─────────────────────────────────────────────────────────────
# WRITE 1 — Daily customer spending
# ─────────────────────────────────────────────────────────────
print("Writing daily customer spending to PostgreSQL...")

try:
    # Step 1 — delete today's existing records
    delete_todays_records("gold.daily_customer_spending")

    # Step 2 — rename columns to match PostgreSQL schema
    spending_pg = daily_spending_df.select(
        F.col("processing_date").cast("date").alias("transaction_date"),
        F.col("customer_id"),
        F.col("bank_code"),
        F.col("total_amount_eur"),
        F.col("transaction_count"),
        F.col("avg_transaction_eur"),
        F.col("max_transaction_eur"),
        F.col("grocery_spend_eur"),
        F.col("ecommerce_spend_eur"),
        F.col("fuel_spend_eur"),
        F.col("entertainment_spend_eur"),
        F.col("transport_spend_eur"),
        F.col("top_category"),
        F.col("pipeline_run_id"),
        F.col("computed_at").alias("created_at")
    )

    # Step 3 — append fresh records
    # mode append — adds rows without touching existing ones
    spending_pg.write.jdbc(
        url=POSTGRES_JDBC_URL,
        table="gold.daily_customer_spending",
        mode="append",
        properties=POSTGRES_PROPS
    )
    print(f"  Written {spending_pg.count()} records")

except Exception as e:
    print(f"  Failed: {e}")

# ─────────────────────────────────────────────────────────────
# WRITE 2 — Fraud indicators
# ─────────────────────────────────────────────────────────────
print()
print("Writing fraud indicators to PostgreSQL...")

try:
    delete_todays_records("gold.fraud_indicators")

    fraud_pg = fraud_df.select(
        F.col("transaction_id"),
        F.col("customer_id"),
        F.col("processing_date").cast("date").alias("transaction_date"),
        F.col("amount_eur"),
        F.col("is_large_amount"),
        F.col("is_suspicious_ip"),
        F.col("is_unknown_merchant"),
        F.col("is_pending_status"),
        F.col("is_unusual_currency"),
        F.col("fraud_score"),
        F.col("risk_level"),
        F.col("pipeline_run_id"),
        F.col("computed_at").alias("created_at")
    )

    fraud_pg.write.jdbc(
        url=POSTGRES_JDBC_URL,
        table="gold.fraud_indicators",
        mode="append",
        properties=POSTGRES_PROPS
    )
    print(f"  Written {fraud_pg.count()} records")

except Exception as e:
    print(f"  Failed: {e}")

# ─────────────────────────────────────────────────────────────
# WRITE 3 — Daily merchant summary
# ─────────────────────────────────────────────────────────────
print()
print("Writing merchant summary to PostgreSQL...")

try:
    delete_todays_records("gold.daily_merchant_summary")

    merchant_pg = merchant_df.select(
        F.col("processing_date").cast("date").alias("transaction_date"),
        F.col("merchant_id"),
        F.col("merchant_name"),
        F.col("merchant_category"),
        F.col("total_revenue_eur"),
        F.col("transaction_count"),
        F.col("unique_customers"),
        F.col("avg_transaction_eur"),
        F.col("pipeline_run_id"),
        F.col("computed_at").alias("created_at")
    )

    merchant_pg.write.jdbc(
        url=POSTGRES_JDBC_URL,
        table="gold.daily_merchant_summary",
        mode="append",
        properties=POSTGRES_PROPS
    )
    print(f"  Written {merchant_pg.count()} records")

except Exception as e:
    print(f"  Failed: {e}")

# ─────────────────────────────────────────────────────────────
# LOG TO MONGODB AUDIT LOG
# ─────────────────────────────────────────────────────────────
print()
print("Logging to MongoDB audit log...")

try:
    client = MongoClient(
        CONFIG["mongodb_uri"],
        serverSelectionTimeoutMS=3000
    )
    db = client[CONFIG["mongodb_db"]]
    db["gold_runs"].insert_one({
        "run_id":           RUN_ID,
        "pipeline_name":    CONFIG["pipeline_name"],
        "processing_date":  CONFIG["processing_date"],
        "spending_records": spending_pg.count(),
        "fraud_records":    fraud_pg.count(),
        "merchant_records": merchant_pg.count(),
        "completed_at":     datetime.utcnow().isoformat(),
        "status":           "SUCCESS"
    })
    client.close()
    print("  Logged to MongoDB")
except Exception as e:
    print(f"  MongoDB logging failed: {e}")

# ─────────────────────────────────────────────────────────────
# VERIFY — Read back from PostgreSQL and confirm
# ─────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("VERIFICATION — reading back from PostgreSQL")
print("=" * 55)

try:
    conn = psycopg2.connect(
        host=CONFIG["postgres_host"],
        port=CONFIG["postgres_port"],
        dbname=CONFIG["postgres_db"],
        user=CONFIG["postgres_user"],
        password=CONFIG["postgres_password"]
    )
    cur = conn.cursor()

    cur.execute("SELECT COUNT(*) FROM gold.daily_customer_spending")
    print(f"daily_customer_spending: {cur.fetchone()[0]} total records")

    cur.execute("SELECT COUNT(*) FROM gold.fraud_indicators")
    print(f"fraud_indicators:        {cur.fetchone()[0]} total records")

    cur.execute("SELECT COUNT(*) FROM gold.daily_merchant_summary")
    print(f"daily_merchant_summary:  {cur.fetchone()[0]} total records")

    cur.execute("""
        SELECT bank_code,
               COUNT(*) as customers,
               ROUND(SUM(total_amount_eur)::numeric, 2) as total_eur
        FROM gold.daily_customer_spending
        GROUP BY bank_code
        ORDER BY total_eur DESC
    """)
    print()
    print("Revenue by bank (from PostgreSQL):")
    print(f"{'Bank':<10} {'Customers':<12} {'Total EUR'}")
    print("-" * 35)
    for row in cur.fetchall():
        print(f"{row[0]:<10} {row[1]:<12} {row[2]}")

    conn.close()

except Exception as e:
    print(f"Verification failed: {e}")

print()
print("=" * 55)
print("Gold aggregation complete")
print(f"Run ID: {RUN_ID}")
print("Power BI can now connect to PostgreSQL")
print("Connection: localhost:5432 / atlas_gold")
print("=" * 55)

Writing daily customer spending to PostgreSQL...
  Deleted 0 existing records from gold.daily_customer_spending
  Written 148 records

Writing fraud indicators to PostgreSQL...
  Deleted 0 existing records from gold.fraud_indicators
  Written 57 records

Writing merchant summary to PostgreSQL...
  Deleted 0 existing records from gold.daily_merchant_summary
  Written 13 records

Logging to MongoDB audit log...
  Logged to MongoDB

VERIFICATION — reading back from PostgreSQL
daily_customer_spending: 148 total records
fraud_indicators:        57 total records
daily_merchant_summary:  13 total records

Revenue by bank (from PostgreSQL):
Bank       Customers    Total EUR
-----------------------------------
MBANK      48           630579.24
PKO        50           608281.54
ING        50           459158.24

Gold aggregation complete
Run ID: 9a1110bb-38e6-4cd8-8014-9b2189af56fb
Power BI can now connect to PostgreSQL
Connection: localhost:5432 / atlas_gold
